In [568]:

import numpy as np
import nnfs
from nnfs.datasets import spiral_data


In [569]:

X,y = spiral_data(100,3)


In [570]:
class Layer_Dense:
  def __init__(self, n_inputs, n_neurons):
    self.weights = 0.10 * np.random.randn(n_inputs, n_neurons)
    self.biases = np.zeros((1, n_neurons))

  def forward(self, inputs):
    self.inputs = inputs
    self.output = np.dot(inputs, self.weights) + self.biases

  def backward(self, dvalues):
    self.dweights = np.dot(self.inputs.T, dvalues)
    self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
    self.dinputs = np.dot(dvalues, self.weights.T)

In [571]:
class Activation_ReLU:
  def forward(self,input):
    self.input = input
    self.output = np.maximum(0,input)

  def backward(self,dvalues):
    self.dinputs = dvalues.copy()
    self.dinputs[self.input <= 0] = 0

In [572]:
class Activation_Softmax:
  def forward(self,inputs):
    exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
    probabilities = exp_values / np.sum(exp_values, axis = 1, keepdims= True)
    self.output = probabilities



In [573]:
class Loss:
  def calculate(self, output, y):
    sample_losses = self.forward(output,y)
    data_loss = np.mean(sample_losses)
    return data_loss

class Loss_CategoricalCrossentropy(Loss):
  def forward(self,y_pred,y_true):
    samples = len(y_pred)
    y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7)

    if len(y_true.shape) == 1:
      correct_confidences = y_pred_clipped[range(samples),y_true]
    elif len(y_true.shape) == 2:
      correct_confidences = np.sum(y_pred_clipped*y_true, axis = 1)

    negative_log_likelihoods = -np.log(correct_confidences)
    return negative_log_likelihoods

In [574]:
class Activation_Softmax_Loss_CategoricalCrossentropy:
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = Loss_CategoricalCrossentropy()

    def forward(self, inputs, y_true):
        self.activation.forward(inputs)
        self.output = self.activation.output
        return self.loss.calculate(self.output, y_true)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs / samples

In [575]:
class Optimizer_SGD:
    def __init__(self, learning_rate=1.0, decay=0.0, momentum=0.0):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.momentum = momentum

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (
                1.0 / (1.0 + self.decay * self.iterations)
            )

    def update_params(self, layer):
        if self.momentum:
            if not hasattr(layer, "weight_momentums"):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.biases)

            weight_updates = (
                self.momentum * layer.weight_momentums
                - self.current_learning_rate * layer.dweights
            )
            layer.weight_momentums = weight_updates

            bias_updates = (
                self.momentum * layer.bias_momentums
                - self.current_learning_rate * layer.dbiases
            )
            layer.bias_momentums = bias_updates
        else:
            weight_updates = -self.current_learning_rate * layer.dweights
            bias_updates = -self.current_learning_rate * layer.dbiases

        layer.weights += weight_updates
        layer.biases += bias_updates

    def post_update_params(self):
        self.iterations += 1

In [576]:
dense1 = Layer_Dense(2, 500)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(500, 3)

loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
optimizer = Optimizer_SGD(learning_rate=0.7, decay=1e-3, momentum=0.90)

In [577]:
for epoch in range(2001):
    # Forward pass
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    data_loss = loss_activation.forward(dense2.output, y)

    predictions = np.argmax(loss_activation.output, axis=1)
    accuracy = np.mean(predictions == y)

    if epoch % 100 == 0:
        print(
            f"epoch {epoch:4d} | "
            f"acc: {accuracy:.3f} | "
            f"loss: {data_loss:.3f} | "
            f"lr: {optimizer.current_learning_rate:.5f}"
        )

    # Backward pass
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # Update params
    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

print("\nTeste final:")
print(f"Loss final: {data_loss:.4f}")
print(f"Acurácia final: {accuracy:.4f}")

epoch    0 | acc: 0.297 | loss: 1.107 | lr: 0.70000
epoch  100 | acc: 0.653 | loss: 0.885 | lr: 0.63694
epoch  200 | acc: 0.683 | loss: 0.662 | lr: 0.58382
epoch  300 | acc: 0.760 | loss: 0.511 | lr: 0.53888
epoch  400 | acc: 0.827 | loss: 0.396 | lr: 0.50036
epoch  500 | acc: 0.883 | loss: 0.301 | lr: 0.46698
epoch  600 | acc: 0.913 | loss: 0.245 | lr: 0.43777
epoch  700 | acc: 0.917 | loss: 0.231 | lr: 0.41201
epoch  800 | acc: 0.930 | loss: 0.204 | lr: 0.38911
epoch  900 | acc: 0.947 | loss: 0.181 | lr: 0.36862
epoch 1000 | acc: 0.943 | loss: 0.170 | lr: 0.35018
epoch 1100 | acc: 0.950 | loss: 0.160 | lr: 0.33349
epoch 1200 | acc: 0.950 | loss: 0.152 | lr: 0.31833
epoch 1300 | acc: 0.953 | loss: 0.146 | lr: 0.30448
epoch 1400 | acc: 0.957 | loss: 0.141 | lr: 0.29179
epoch 1500 | acc: 0.957 | loss: 0.137 | lr: 0.28011
epoch 1600 | acc: 0.957 | loss: 0.133 | lr: 0.26933
epoch 1700 | acc: 0.957 | loss: 0.129 | lr: 0.25936
epoch 1800 | acc: 0.957 | loss: 0.126 | lr: 0.25009
epoch 1900 |